In [ ]:
%%capture
!pip install transformers
!pip install tqdm
!pip install python-dotenv
!pip install accelerate
!pip install --upgrade transformers
!pip install -U bitsandbytes>=0.46.1
!pip install peft


In [2]:
import os
import glob

from datasets import load_dataset
from dotenv import load_dotenv
import torch
import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
import json

load_dotenv()

QUANTIZATIONS = {
    "v0": None, # No quantization
    "v1": {
        "load_in_4bit": True,
        "bnb_4bit_use_double_quant": False,
        "bnb_4bit_quant_type": "nf4",
        "modules_to_skip": ["vision_tower", "multi_modal_projector"]
    },
}

def format_message(question: str, image, prompt: str = "Answer briefly") -> list:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"{prompt}: {question}"}
            ]
        }
    ]
    return messages


def get_quantization_config(modules_to_skip: list[str] | None = None, **kwargs) -> BitsAndBytesConfig:
    if modules_to_skip is None:
        modules_to_skip = []
    else:
        # lm_head MUST be always here, since llm_int8_skip_modules clean the default, see open PR issue
        # https://github.com/huggingface/transformers/issues/45674
        # e.g. modules_to_skip = ["lm_head", "vision_tower", "multi_modal_projector"]
        modules_to_skip.append("lm_head")

    default_kwargs = {
        "load_in_4bit": True,
        "bnb_4bit_use_double_quant": False,
        "bnb_4bit_compute_dtype": torch.bfloat16,
        "bnb_4bit_quant_type": "nf4",
    }
    default_kwargs.update(kwargs)

    quantization_config = BitsAndBytesConfig(
        llm_int8_skip_modules=modules_to_skip,
        **default_kwargs
    )

    return quantization_config


def load_model_and_processor(model_id: str, quantization_config=None):
    if quantization_config is None:
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
    else:
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=quantization_config,
        )
    processor = AutoProcessor.from_pretrained(model_id)
    # Set padding side to left for batch generation
    processor.tokenizer.padding_side = "left"
    return model, processor


def load_data_from_disk(dataset_path: str, files_regex: str, split_name: str = "validation"):
    files = glob.glob(os.path.join(dataset_path, "**", "*.arrow"), recursive=True)
    val_files = [f for f in files if files_regex in f]
    data = load_dataset(
        "arrow",
        data_files={split_name: val_files},
        split=split_name,
    )
    return data


def load_data_from_hf(dataset_name: str = "lmms-lab/textvqa", split: str = "validation"):
    data = load_dataset(dataset_name, split=split, streaming=True)
    return data


def generate_and_decode(model, processor, inputs):
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=100, pad_token_id=processor.tokenizer.eos_token_id)
        # Trim out the prompt tokens to decode only the predictions
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]
    return output_text

In [3]:
# model_id = "Qwen/Qwen3.5-0.8B"
model_id = "Qwen/Qwen3.5-2B"
LOCAL = False
prompt = "Answer briefly based on the image"
quantization = "v1" # v0 is no quantization
out_path = f"/content/drive/MyDrive/LLM_results/{model_id.split("/")[1]}_{quantization}/textvqa.json"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

quant_params = QUANTIZATIONS[quantization]
quant_config = get_quantization_config(**quant_params) if quant_params else None
model, processor = load_model_and_processor(model_id, quantization_config=quant_config)

if LOCAL:
    dataset_path = os.getenv("TEXTVQA_DATA_PATH")
    if not dataset_path:
        raise ValueError("Please set the TEXTVQA_DATA_PATH environment variable to the path of the TextVQA dataset.")
    data = load_data_from_disk(dataset_path, "validation-00000")
else:
    data = load_data_from_hf()

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [4]:
batch_size = 4
n_samples = 1000
results = []

# Convert streaming dataset to an iterator for manual slicing
data_iter = iter(data)

# Progress bar for batches
for i in tqdm.tqdm(range(0, n_samples, batch_size), desc="Processing batches"):
    batch_samples = []
    try:
        for _ in range(batch_size):
            batch_samples.append(next(data_iter))
    except StopIteration:
        break

    # Prepare inputs for the batch
    batch_questions = [s["question"] for s in batch_samples]
    batch_images = [s["image"] for s in batch_samples]

    # Format messages for each sample in batch
    batch_messages = [format_message(q, img, prompt=prompt) for q, img in zip(batch_questions, batch_images)]
    batch_texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]

    # Processor handles padding across the batch automatically
    inputs = processor(
        text=batch_texts,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate for the whole batch
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50, pad_token_id=processor.tokenizer.eos_token_id)

        # Trim prompts and decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        batch_outputs = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

    # Collect results
    for j, sample in enumerate(batch_samples):
        results.append({
            "question_id": sample["question_id"],
            "question": sample["question"],
            "answers": sample["answers"],
            "predicted_answer": batch_outputs[j].strip(),
        })

# Save results
with open(out_path, "w") as f:
    json.dump(results, f, indent=4)
print(f"Done! Processed {len(results)} samples.")

Processing batches: 100%|██████████| 250/250 [07:22<00:00,  1.77s/it]

Done! Processed 1000 samples.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from torch.optim import AdamW

# LoRA training config
n_train_samples = 500
train_batch_size = 2
learning_rate = 2e-4
lora_out_path = f"/content/drive/MyDrive/LLM_results/{model_id.split('/')[1]}_{quantization}_lora"

# Prepare quantized model for gradient computation
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

# Load train split
train_data = load_data_from_hf(split="train")
train_iter = iter(train_data)

optimizer = AdamW(lora_model.parameters(), lr=learning_rate)
lora_model.train()

total_loss = 0.0
num_batches = 0

for i in tqdm.tqdm(range(0, n_train_samples, train_batch_size), desc="LoRA SFT"):
    batch_samples = []
    try:
        for _ in range(train_batch_size):
            batch_samples.append(next(train_iter))
    except StopIteration:
        break

    batch_questions = [s["question"] for s in batch_samples]
    batch_images = [s["image"] for s in batch_samples]
    batch_answers = [s["answers"][0] for s in batch_samples]

    # Full conversation messages (user + assistant)
    batch_messages_full = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": f"{prompt}: {q}"}
                ]
            },
            {"role": "assistant", "content": ans}
        ]
        for q, img, ans in zip(batch_questions, batch_images, batch_answers)
    ]

    # Prompt-only messages (for masking)
    batch_messages_prompt = [[msgs[0]] for msgs in batch_messages_full]

    batch_texts_full = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch_messages_full
    ]
    batch_texts_prompt = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in batch_messages_prompt
    ]

    # Tokenize full sequences with left-padding
    inputs = processor(
        text=batch_texts_full,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Build labels: mask padding and prompt tokens, keep only answer tokens
    labels = inputs["input_ids"].clone()
    for idx in range(len(batch_samples)):
        # Tokenize prompt individually (with image) to get prompt token length
        prompt_inputs = processor(
            text=[batch_texts_prompt[idx]],
            images=[batch_images[idx]],
            return_tensors="pt"
        )
        prompt_len = prompt_inputs["input_ids"].shape[1]

        # Offset for left-padding in the batched sequence
        pad_len = (inputs["attention_mask"][idx] == 0).sum().item()

        # Mask padding + prompt tokens
        labels[idx, : pad_len + prompt_len] = -100

    optimizer.zero_grad()
    outputs = lora_model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    num_batches += 1

    if num_batches % 50 == 0:
        print(f"Step {num_batches}, Avg Loss: {total_loss / num_batches:.4f}")

print(f"Training complete. Final avg loss: {total_loss / num_batches:.4f}")

# Save LoRA adapter and processor
os.makedirs(lora_out_path, exist_ok=True)
lora_model.save_pretrained(lora_out_path)
processor.save_pretrained(lora_out_path)
print(f"LoRA adapter saved to {lora_out_path}")
